# DOD-VLM End-to-End — Described Object Detection

> **Mục tiêu:** chạy **Run All** từ một môi trường Python có PyTorch CUDA đến một hệ thống DOD-VLM đã train, calibrate, đánh giá, đóng gói và reload để inference thật.  
> **Máy mục tiêu:** Dell Precision + NVIDIA RTX A3000 12 GB.  
> **Input:** ảnh RGB + mô tả tự do bằng tiếng Anh.  
> **Output:** 0, 1 hoặc N bounding boxes thỏa toàn bộ mô tả.

Kiến trúc được khóa theo kế hoạch đã thống nhất:

```text
Ảnh ──► YOLOE-26s prompt-free ──► K proposal độc lập query
  └──► Frozen MobileNetV3-FPN + RoIAlign ──► object tokens

Câu ──► DistilBERT ──► full token features + target/anchor phrase features

object tokens + text
       ├── A0: object self-attention + full-sentence cross-attention
       └── A1: entity matching + latent anchor + pairwise relation geometry
                          │
                    learned fusion gate
                          │
             candidate logits + explicit null head
                          │
               temperature + threshold calibration
                          │
                       0 / 1 / N boxes
```

### Cam kết vận hành của notebook

- Session rõ ràng, Markdown tiếng Việt và progress/ETA bằng `tqdm`.
- Cache proposal/object feature theo từng ảnh; chạy lại không tính lại ảnh đã hoàn tất.
- `recovery.pt` giữa epoch, `last.pt` mỗi epoch và `best.pt` theo validation.
- Preflight forward/backward thật trên dữ liệu cache và tự giảm batch khi A3000 thiếu VRAM.
- Train và inference dùng **cùng query parser**, không đưa GT category vào input của A1.
- Trường hợp YOLOE không có proposal dùng context fallback token để tránh attention all-padding; token fallback không bao giờ được trả thành box.
- Artifact cuối **self-contained ở mức model assets**: tokenizer, trained core, YOLOE checkpoint và frozen region encoder đều nằm trong cùng thư mục/ZIP; runtime không tải model weights từ Internet.
- Sau export, notebook giải phóng GPU, reload artifact từ disk, infer một ảnh thật và benchmark latency P50/P95.


## Session 0 — Preflight môi trường

Notebook chỉ tự cài/upgrade các thư viện Python nhẹ. **Không tự thay PyTorch** vì cài nhầm wheel CPU/CUDA có thể phá môi trường đang chạy. PyTorch + TorchVision phải là cặp CUDA tương thích; cell này kiểm tra version, CUDA và GPU trước khi đi tiếp.

In [ ]:
import importlib.metadata as metadata
import subprocess
import sys
from pathlib import Path
import shutil

# packaging cần để so sánh version trước khi import các package lớn.
try:
    from packaging.version import Version
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "packaging>=24.0"])
    from packaging.version import Version

REQUIRED = {
    "numpy": ("1.26", "numpy>=1.26,<3"),
    "pandas": ("2.2", "pandas>=2.2,<3"),
    "Pillow": ("10.0", "Pillow>=10,<12"),
    "requests": ("2.32", "requests>=2.32,<3"),
    "tqdm": ("4.66", "tqdm>=4.66,<5"),
    "huggingface-hub": ("0.35", "huggingface-hub>=0.35,<2"),
    "transformers": ("4.46", "transformers>=4.46,<6"),
    "ultralytics": ("8.4.0", "ultralytics>=8.4.0,<9"),
    "matplotlib": ("3.9", "matplotlib>=3.9,<4"),
}

to_install = []
for dist_name, (minimum, spec) in REQUIRED.items():
    try:
        current = Version(metadata.version(dist_name))
    except metadata.PackageNotFoundError:
        current = None
    if current is None or current < Version(minimum):
        to_install.append(spec)

if to_install:
    print("Đang cài/upgrade dependency:", to_install)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *to_install])

for dist_name in ["torch", "torchvision"]:
    try:
        print(f"{dist_name}: {metadata.version(dist_name)}")
    except metadata.PackageNotFoundError as exc:
        raise RuntimeError(
            f"Thiếu {dist_name}. Hãy cài cặp PyTorch/TorchVision CUDA chính thức trước rồi Run All lại."
        ) from exc

import torch
import torchvision

if not torch.cuda.is_available():
    raise RuntimeError(
        "PyTorch hiện không thấy CUDA. Cần cài PyTorch CUDA phù hợp NVIDIA driver của máy A3000."
    )

torch_v = Version(torch.__version__.split("+")[0])
vision_v = Version(torchvision.__version__.split("+")[0])
if torch_v.major == 2 and vision_v.major == 0:
    expected_vision_minor = torch_v.minor + 15
    if vision_v.minor != expected_vision_minor:
        raise RuntimeError(
            f"Torch {torch_v} và TorchVision {vision_v} có vẻ không cùng release line "
            f"(với torch 2.{torch_v.minor}, thường cần torchvision 0.{expected_vision_minor})."
        )

print("CUDA runtime:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB")

# Tìm repo root bất kể notebook được mở từ root hay thư mục notebooks/.
cwd = Path.cwd().resolve()
if (cwd / "dod_vlm").is_dir():
    REPO_ROOT = cwd
elif (cwd.parent / "dod_vlm").is_dir():
    REPO_ROOT = cwd.parent
else:
    raise RuntimeError("Không tìm thấy package dod_vlm/. Hãy mở notebook từ repository đã pull đầy đủ.")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

free_gb = shutil.disk_usage(REPO_ROOT).free / 2**30
print("Repo root:", REPO_ROOT)
print(f"Disk free: {free_gb:.1f} GB")
if free_gb < 30:
    print("⚠ Khuyến nghị có ít nhất ~30 GB trống cho ảnh COCO + feature cache + checkpoints.")

print("✓ Session 0 preflight PASS")

## Session 1 — Cấu hình, workspace và cơ chế resume

Mọi output nằm trong `dod_vlm_workspace/`. `RunState` ghi tiến độ từng stage; file cache/checkpoint được ghi atomic (`.tmp` → rename) để giảm nguy cơ hỏng file nếu kernel/máy bị ngắt.

In [ ]:
from dataclasses import asdict
import gc
import json
import random
import time

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

from dod_vlm import Config, Paths, RunState, gpu_report, seed_everything, stage_timer
from dod_vlm.utils import atomic_json_dump

cfg = Config()
paths = Paths.from_config(cfg)
state = RunState(cfg, paths)
seed_everything(cfg.seed)

torch.backends.cudnn.benchmark = True
DEVICE = torch.device("cuda")

print(json.dumps(asdict(cfg), indent=2, ensure_ascii=False))
print("Workspace:", paths.root)
print("Config hash:", cfg.signature())
gpu_report("Khởi tạo:")

## Session 2 — Tải và chuẩn hóa gRefCOCO

Dùng gRefCOCO làm tập train/validation/test chính vì có **single-target, multi-target và no-target**. Annotation được chuyển về schema query-level thống nhất. `target_categories` vẫn được giữ để chẩn đoán, nhưng **không được đưa vào input model**.

In [ ]:
from dod_vlm.data import download_annotations, build_manifest

refs_path, instances_path = download_annotations(cfg, paths, state)
manifest, manifest_path, image_meta_path = build_manifest(
    cfg, paths, state, refs_path, instances_path
)

print("Manifest:", manifest_path)
display(
    manifest.groupby("split").agg(
        samples=("sample_id", "size"),
        images=("image_id", "nunique"),
        no_target=("no_target", "sum"),
    )
)
display(manifest.head(3))

## Session 3 — Tải các ảnh COCO thực sự cần dùng

Không tải mù toàn bộ COCO. Mỗi ảnh được verify bằng PIL; file dở dang dùng hậu tố `.part` và tự retry. Chạy lại notebook sẽ skip ảnh hợp lệ đã có.

In [ ]:
from dod_vlm.data import download_images

download_images(cfg, paths, state, image_meta_path)

## Session 4 — Prompt-independent proposals và object-token cache

- YOLOE-26s prompt-free tạo candidate không phụ thuộc câu.
- Frozen FasterRCNN-MobileNetV3-FPN dùng **đúng `GeneralizedRCNNTransform` pretrained**, sau đó MultiScaleRoIAlign theo YOLOE boxes.
- Nếu YOLOE không trả box, tạo một full-image **context fallback token** để attention luôn có token hợp lệ; fallback không được tính loss candidate và không bao giờ xuất box.
- ROI cho GT train được cache riêng để optional GT injection không phải chạy backbone lại.
- YOLOE checkpoint và frozen region encoder được lưu local để artifact cuối không phụ thuộc model download.

In [ ]:
from dod_vlm.vision import cache_visual_features

feature_dir, proposal_local, region_local, cache_signature = cache_visual_features(
    cfg, paths, state, manifest, DEVICE
)
print("Feature cache:", feature_dir)
print("Proposal local:", proposal_local)
print("Region encoder local:", region_local)
gpu_report("Session 4:")

## Session 5 — Proposal diagnostics + kiểm tra query parser

Diagnostics này **không chặn pipeline**, nhưng cho biết localization ceiling trước khi reasoning. gRefCOCO không có annotation anchor-role đầy đủ cho mọi câu, nên notebook chỉ báo target proposal recall; anchor/joint recall cần một relation-diagnostic set riêng.

Quan trọng: `parse_query()` bên dưới là **chính xác cùng hàm** được Dataset dùng khi train và Runtime dùng khi infer.

In [ ]:
from dod_vlm.query import parse_query
from dod_vlm.evaluation import proposal_diagnostics

examples = [
    "find all red cars",
    "find the car next to the bench",
    "the cup to the left of the red box",
]
for text in examples:
    print(text, "->", parse_query(text))

proposal_metrics = proposal_diagnostics(cfg, manifest, feature_dir, split="val")
print(json.dumps(proposal_metrics, indent=2))
atomic_json_dump(proposal_metrics, paths.run / "proposal_diagnostics.json")
state.mark("proposal_diagnostics_complete", metrics=proposal_metrics)

## Session 6 — Train DOD-VLM với preflight VRAM và checkpoint/resume

`train_model()` thực hiện tuần tự:

1. load DistilBERT pretrained và chỉ mở 2 layer cuối;
2. chạy **forward + backward thật** trên batch từ cache;
3. nếu CUDA OOM, tự giảm batch `8 → 6 → 4 → 2` và tăng gradient accumulation để giữ effective batch xấp xỉ 24;
4. train A0 + A1 + fusion gate + explicit null head;
5. focal BCE candidate + null BCE + entity auxiliary loss;
6. lưu `recovery.pt` định kỳ **chỉ sau optimizer step**; `last.pt` sau epoch; `best.pt` theo validation F1;
7. thứ tự DataLoader deterministic theo epoch và GT injection deterministic theo `sample_id`, giúp resume giữa epoch ổn định hơn.

In [ ]:
from dod_vlm.engine import train_model

model, tokenizer, batch_size, grad_accum, best_ckpt, last_ckpt = train_model(
    cfg,
    paths,
    state,
    manifest,
    feature_dir,
    cache_signature,
    DEVICE,
)
print("Chosen batch size:", batch_size)
print("Gradient accumulation:", grad_accum)
print("Best checkpoint:", best_ckpt)

## Session 7 — Temperature calibration, threshold calibration và final evaluation

Validation logits chỉ chạy model **một lần**. Sau đó:

- fit candidate temperature và null temperature bằng BCE;
- grid-search candidate/null threshold trên CPU;
- khóa calibration;
- evaluate tất cả held-out split có sẵn (`test`, `testA`, `testB`);
- báo set Precision/Recall/F1@0.5, AP50, AP75, mAP@0.5:0.95, negative reject accuracy, false boxes/negative, null Brier và null ECE.

In [ ]:
from dod_vlm.evaluation import evaluate_and_calibrate

calibration, final_metrics, _ = evaluate_and_calibrate(
    cfg,
    paths,
    state,
    manifest,
    feature_dir,
    model,
    tokenizer,
    best_ckpt,
    batch_size,
    DEVICE,
)
print("Calibration:")
print(json.dumps(calibration, indent=2))
print("Final metrics:")
print(json.dumps(final_metrics, indent=2))

## Session 8 — Đóng gói artifact self-contained

Không coi một `state_dict` đơn lẻ là deployment artifact. Thư mục `final_model/` chứa:

```text
final_model/
├── dod_vlm_core.pt       # trained multimodal core + DistilBERT weights/config
├── tokenizer/            # tokenizer local
├── proposal_model.pt     # YOLOE prompt-free local checkpoint
├── region_encoder.pt     # frozen MobileNetV3-FPN state + transform recipe
├── config.json
├── calibration.json
├── metrics.json
└── README.txt
```

Notebook đồng thời tạo `dod_vlm_final_bundle.zip`. Runtime chỉ cần Python dependencies + thư mục/ZIP đã giải nén, không cần tải lại model weights từ mạng.

In [ ]:
from dod_vlm.runtime import export_final_artifact

artifact_dir = export_final_artifact(
    cfg,
    paths,
    state,
    model,
    tokenizer,
    calibration,
    final_metrics,
    proposal_local,
    region_local,
    cache_signature,
)
print("Artifact directory:", artifact_dir)
print("ZIP:", paths.run / "dod_vlm_final_bundle.zip")

## Session 9 — Giải phóng VRAM, reload artifact, inference thật và benchmark latency

Đây là kiểm tra cuối: xóa training model khỏi GPU, tạo runtime **mới hoàn toàn từ disk**, chạy một query thật, rồi benchmark end-to-end latency. Nếu cell này hoàn tất, artifact đã chứng minh được vòng save → reload → inference.

In [ ]:
# Giải phóng object training trước khi load runtime mới để tránh double-allocation trên A3000 12 GB.
del model
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
gpu_report("Trước reload runtime:")

from dod_vlm.runtime import reload_and_smoke_test

runtime, demo_result, runtime_metrics = reload_and_smoke_test(
    cfg,
    paths,
    state,
    artifact_dir,
    manifest,
    DEVICE,
)
gpu_report("Runtime:")

### Trực quan hóa prediction sanity check

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

image = Image.open(demo_result["image"]).convert("RGB")
fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(image)
for pred in demo_result["prediction"]:
    x1, y1, x2, y2 = pred["box"]
    ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, linewidth=2))
    ax.text(x1, max(0, y1 - 4), f'{pred["score"]:.2f} {pred["proposal_label"]}', fontsize=9)
ax.set_title(demo_result["query"])
ax.axis("off")
plt.show()

# Hoàn tất

Nếu **Run All** đi đến đây không lỗi, bạn có:

- cache/resume đầy đủ;
- `best.pt` và `last.pt` phục vụ tiếp tục nghiên cứu;
- calibration + final metrics trên held-out split;
- `final_model/` self-contained về model assets;
- `dod_vlm_final_bundle.zip` để mang sang máy triển khai;
- runtime đã reload và infer thật;
- latency P50/P95 của pipeline PyTorch hiện tại.

> **Lưu ý nghiên cứu:** mục tiêu ≥10 FPS trên edge/Jetson là bước tối ưu deployment tiếp theo. Notebook này đo latency thật nhưng không giả định rằng PyTorch research runtime đã đạt 10 FPS. Nếu chưa đạt, tối ưu TensorRT/engine là một phase riêng sau khi quality đã được xác nhận.